In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
db_user = "postgres"
db_password = "naya_password123"
db_host = "localhost"
db_port = "5432"
db_name = "ecommerce_retention_db"

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [3]:
query = "SELECT COUNT(*) FROM customers;"
test_df = pd.read_sql(query, engine)
print(test_df)

   count
0  99441


# EDA 

In [4]:
# Load core tables into pandas DataFrames
customers_df = pd.read_sql("SELECT * FROM customers;", engine)
orders_df = pd.read_sql("SELECT * FROM orders;", engine)
payments_df = pd.read_sql("SELECT * FROM order_payments;", engine)

print("Customers shape:", customers_df.shape)
print("Orders shape:", orders_df.shape)
print("Payments shape:", payments_df.shape)

Customers shape: (99441, 5)
Orders shape: (99441, 8)
Payments shape: (103886, 5)


In [5]:
print("Missing values in customers_df:\n", customers_df.isnull().sum())
print("\nMissing values in orders_df:\n", orders_df.isnull().sum())
print("\nMissing values in payments_df:\n", payments_df.isnull().sum())

Missing values in customers_df:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing values in orders_df:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Missing values in payments_df:
 order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


## order_status values: orders "delivered"  vs "cancelled"/other

In [6]:
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## customers & orders ---> join ---> master dataframe  (only delivered orders )

In [7]:
# Filter only delivered orders
delivered_orders = orders_df[orders_df['order_status'] == 'delivered'].copy()

# Merge with customers to get customer_unique_id
orders_customers = delivered_orders.merge(customers_df, on='customer_id', how='left')

# Merge with payments to get total payment value per order
# (an order can have multiple payment rows, so we sum them first)
order_totals = payments_df.groupby('order_id')['payment_value'].sum().reset_index()

master_df = orders_customers.merge(order_totals, on='order_id', how='left')

print("Master dataframe shape:", master_df.shape)
master_df.head()

Master dataframe shape: (96478, 13)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62


## RFM Analysis — real retention/CLV works start from here 

#### RFM ka matlab hai:

#### Recency — customer ne kitne din pehle last order kiya
#### Frequency — customer ne kitni baar order kiya (total)
#### Monetary — customer ne total kitna spend kiya

In [8]:
# Yeh teeno milke customer ko segments mein daalne mein madad karte hain 
# (jaise "high-value loyal customers" vs "one-time buyers at risk of churn") 
# — yehi tumhara core business insight banega.

In [9]:
# reference_date — hum dataset ke sabse recent order ke agle din ko "today" maan rahe hain 
# (kyunki yeh purana historical data hai, actual aaj ki date se recency calculate karna galat hoga).
# groupby('customer_unique_id') — yeh crucial hai, isi se hum real customers ke level pe aggregate kar rahe hain, na ki order-level pe.
# recency — last purchase se reference date tak kitne din.
# frequency — unique orders ka count.
# monetary — total kitna spend kiya.

In [10]:
# Reference date: one day after the last purchase date in the dataset
reference_date = master_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print("Reference date for Recency calculation:", reference_date)

# Group by customer_unique_id (the REAL person, not order-level customer_id)
rfm = master_df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

print("RFM shape:", rfm.shape)
rfm.head()

Reference date for Recency calculation: 2018-08-30 15:00:00
RFM shape: (93358, 4)


,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
2,0000f46a3911fa3c0805444483337064,537,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [11]:
# RFM table ban gaya — 93,358 unique customers (96,478 delivered orders se, matlab kuch customers ne multiple orders kiye,
# jo count kam hone se confirm hota hai — 93,358 < 96,478, exactly jaisa expect kiya tha).

In [12]:
# pehle 5 rows mein sabki frequency = 1 hai (matlab yeh sab one-time buyers hain). 
# Yeh Olist dataset ki ek known characteristic hai — is marketplace mein repeat customers kaafi kam hote hain 
# (main isko exact % ke saath confirm nahi kar sakta abhi, but tumhe khud yeh number nikal ke dekhna chahiye
# — yeh tumhare dashboard ka sabse bada insight ban sakta hai).

## Check  —  % age of customers which are repeat buyers

In [13]:
repeat_customers = (rfm['frequency'] > 1).sum()
total_customers = rfm.shape[0]
repeat_pct = (repeat_customers / total_customers) * 100

print(f"Total customers: {total_customers}")
print(f"Repeat customers (2+ orders): {repeat_customers}")
print(f"Repeat purchase rate: {repeat_pct:.2f}%")

Total customers: 93358
Repeat customers (2+ orders): 2801
Repeat purchase rate: 3.00%


In [14]:
# Matlab 93,358 customers mein se sirf 2,801 hi doosri baar order karte hain. 
# Yeh number kaafi kam hai (main exact industry-benchmark comparison nahi kar sakta bina current data verify kiye,
# lekin general e-commerce standards ke hisaab se yeh clearly low hai)

# headline insight "Olist marketplace mein customer retention ek critical problem hai — 97% customers sirf ek hi baar purchase karte hain."

In [15]:
# "Olist marketplace mein customer retention ek major challenge hai, 
# aur yeh dashboard identify karta hai ki kaunse segments retention ke liye sabse zyada potential rakhte hain."

## RFM Segments — which customers are "at risk" & which are "champions"

In [16]:
# Ab hum har customer ko score denge (1-4 scale, quartile-based) aur unhe meaningful business segments mein daalenge.

In [17]:
# Score each metric into quartiles (1 = worst, 4 = best)
# Note: for recency, LOWER is better (recent = good), so we reverse the labels
rfm['R_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4])

rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

rfm.head(10)

,customer_unique_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,3,413
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3,1,1,311
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214
5,0004bd2a26a76fe21f786e4fbd80607f,146,1,166.98,3,1,3,313
6,00050ab1314c0e55a6ca13cf7181fecf,132,1,35.38,3,1,1,311
7,00053a61a98854899e70ed204dd4bafe,183,1,419.18,3,1,4,314
8,0005e1862207bf6ccc02e4228effd9a0,543,1,150.12,1,1,3,113
9,0005ef4cd20d2893f0d9fbd94d3c0d97,170,1,129.76,3,1,3,313


In [18]:
# pd.qcut data ko 4 barabar hisso (quartiles) mein baant deta hai — top 25%, next 25%, waghera.

# Recency ke liye humne labels reverse kiye ([4,3,2,1] instead of [1,2,3,4]) — kyunki kam recency (matlab customer ne recently order kiya)
# achhi baat hai, isliye usse high score (4) milna chahiye.

# Frequency ke liye .rank(method='first') use kiya hai kyunki bahut saare customers ki frequency same hai (1 order)
# — bina rank ke qcut error de sakta hai duplicate edges ki wajah se.

# RFM_score teeno scores ko jodkar ek 3-digit code banata hai (jaise "444" = best customer, "111" = worst).

In [19]:
# RFM scores ban gaye — R, F, M sab quartiles mein sahi se assign ho gaye hain, aur RFM_score 3-digit code bhi dikh raha hai 
# (jaise "413", "311"). Tumne notice kiya hoga ki F_score mein zyadatar 1 hi dikh raha hai — yeh expected hai kyunki humne pehle dekha,
# 97% customers sirf 1 order karte hain, isliye frequency quartile mein woh sab lowest bucket mein aa jaate hain.

## Make named business segments (such as "Champions", "At Risk", "Lost")

In [20]:
# Ab hum RFM scores ko real business labels mein convert karenge, jo dashboard mein directly dikhengi 
# (kyunki koi bhi stakeholder "RFM score 413" nahi samjhega, lekin "At Risk" turant samajh jayega).

In [21]:
def segment_customer(row):
    r = int(row['R_score'])
    f = int(row['F_score'])
    m = int(row['M_score'])
    
    if r >= 3 and f >= 2:
        return 'Loyal / Repeat Customers'
    elif r >= 3 and f == 1 and m >= 3:
        return 'Recent High-Value (New Champions)'
    elif r >= 3 and f == 1:
        return 'New Customers'
    elif r == 2:
        return 'At Risk'
    elif r == 1 and m >= 3:
        return 'Lost High-Value (Win-Back Priority)'
    else:
        return 'Lost / Churned'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

rfm['segment'].value_counts()

segment
Loyal / Repeat Customers               35170
At Risk                                23200
Lost / Churned                         12069
Lost High-Value (Win-Back Priority)    11242
New Customers                           5850
Recent High-Value (New Champions)       5827
Name: count, dtype: int64

In [22]:
# Loyal/Repeat — recent aur multiple orders — best customers.

# Recent High-Value (New Champions) — abhi recently ek hi high-value order kiya — potential future loyal customer.

# New Customers — recent, ek order, average spend.

# At Risk — beech ki recency — na bahut recent, na bahut purana — abhi bhi bacha sakte ho.

# Lost High-Value (Win-Back Priority) — bahut purana order, lekin unhone kabhi high spend kiya tha — win-back campaign ka best target.

# Lost/Churned — purana, low value — priority nahi.

In [23]:
# result mein ek problem hai jo humein pehle fix karni chahiye, warna dashboard galat insight dega.

# Issue : "Loyal / Repeat Customers: 35,170" dikh raha hai — lekin humne abhi Step 9 mein confirm kiya tha ki
# sirf 2,801 customers (3%) ne 2 ya usse zyada orders kiye hain. Toh "Loyal/Repeat" segment mein 35,170 customers kaise aa gaye?

# Reason: F_score calculate karne mein maine rank(method='first') use kiya tha taaki duplicate values ka error na aaye. 
# Lekin problem yeh hai — chunki 97% customers ki frequency exactly 1 hai, rank(method='first') inhe sirf unke row-order 
# ke hisaab se arbitrary ranks de raha hai (1, 2, 3...), aur phir qcut un ranks ko 4 quartiles mein baant raha hai. 
# Matlab: same frequency (1) wale customers ko randomly alag-alag F_scores (1, 2, 3, ya 4) mil rahe hain — jo galat hai, 
# kyunki unki actual frequency same hai.

# Yeh ek classic data issue hai jab ek column mein bahut zyada values same hoti hain (yahan 97% customers ki frequency=1) 
# — standard quartile-based scoring is case mein kaam nahi karta.

In [24]:
# Fix: frequency scoring — since 97% customers have frequency=1, 
# quartile-based scoring doesn't work. Use a simple tiered approach instead.
def score_frequency(f):
    if f == 1:
        return 1
    elif f == 2:
        return 2
    elif f in [3, 4]:
        return 3
    else:
        return 4

rfm['F_score'] = rfm['frequency'].apply(score_frequency)

# Recalculate RFM_score and segment with corrected F_score
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['segment'] = rfm.apply(segment_customer, axis=1)

rfm['segment'].value_counts()

# "Loyal / Repeat Customers" ka count bahut kam ho jayega (2,801 ke aas-paas ya usse thoda kam/zyada, depending on kitne repeat 
# customers recent bhi hain), aur baaki segments (khaaskar "New Customers" aur "Lost/Churned") mein badi shift dikhegi — kyunki 
# ab yeh sahi tarike se reflect karega ki zyadatar customers one-time buyers hain.

segment
At Risk                                23200
New Customers                          22756
Recent High-Value (New Champions)      22542
Lost / Churned                         12069
Lost High-Value (Win-Back Priority)    11242
Loyal / Repeat Customers                1549
Name: count, dtype: int64

In [25]:
# Ab numbers logically sahi lag rahe hain. Total check karte hain: 23,200 + 22,756 + 22,542 + 12,069 + 11,242 + 1,549 = 
# 93,358 ✓ — match ho raha hai total customers se, koi row miss nahi hui.

# "Loyal / Repeat Customers" ab 1,549 hai (2,801 repeat customers mein se) — yeh bhi sahi hai, kyunki "Loyal" banne ke liye humne
# condition rakhi thi recent bhi ho AND repeat bhi ho (r≥3 and f≥2). Baaki 2,801-1,549 = 1,252 repeat customers aise honge jinhone
# repeat toh kiya lekin recently nahi (woh "At Risk" ya "Lost" mein chale gaye) — yeh business sense mein bilkul theek hai
# : purana repeat customer bhi churn kar sakta hai.

# Data ab clean aur trustworthy hai. Great catch fix karne mein saath diya — yeh exactly wahi tarah ka QA hai jo 
# interviewer ko impress karta hai agar tum explain karo ki tumne quartile-scoring ka issue kaise identify aur fix kiya.

## Save RFM table back to postgre SQL 

In [26]:
# Ab hum yeh poora rfm dataframe (customer_unique_id, recency, frequency, monetary, scores, segment) ek nayi PostgreSQL table 
# mein save karenge — taaki Power BI seedha database se connect karke isse use kar sake.

rfm.to_sql('customer_rfm_segments', engine, if_exists='replace', index=False)
print("RFM table saved to PostgreSQL as 'customer_rfm_segments'")

RFM table saved to PostgreSQL as 'customer_rfm_segments'


In [27]:
# to_sql() pandas dataframe ko directly PostgreSQL mein ek naya table bana dega, if_exists='replace' matlab agar table pehle se 
# ho toh overwrite kar dega (useful hai jab tum baad mein RFM logic improve karo aur re-run karo).

## Python mein broader KPIs ke liye data taiyar karo

In [28]:
# Power BI kholne se pehle, hum Python mein kuch aur aggregated tables banayenge jo dashboard ke Page 1 (overview) ke liye chahiye honge.
# Monthly revenue trend
master_df['order_month'] = master_df['order_purchase_timestamp'].dt.to_period('M').astype(str)
monthly_revenue = master_df.groupby('order_month')['payment_value'].sum().reset_index()
monthly_revenue.columns = ['order_month', 'total_revenue']

# Top product categories by revenue (needs order_items + products)
order_items_df = pd.read_sql("SELECT * FROM order_items;", engine)
products_df = pd.read_sql("SELECT * FROM products;", engine)
category_translation_df = pd.read_sql("SELECT * FROM product_category_translation;", engine)

category_revenue = order_items_df.merge(products_df, on='product_id', how='left') \
    .merge(category_translation_df, on='product_category_name', how='left') \
    .groupby('product_category_name_english')['price'].sum() \
    .reset_index().sort_values('price', ascending=False)
category_revenue.columns = ['category', 'total_revenue']

# Revenue by state
state_revenue = master_df.groupby('customer_state')['payment_value'].sum().reset_index().sort_values('payment_value', ascending=False)
state_revenue.columns = ['state', 'total_revenue']

print("Monthly revenue rows:", monthly_revenue.shape)
print("Category revenue rows:", category_revenue.shape)
print("State revenue rows:", state_revenue.shape)

monthly_revenue.head()

# 3 aggregations :

# Monthly revenue — time trend line chart ke liye (business growth/seasonality dikhane ke liye).
# Category revenue — kaunsi product categories sabse zyada revenue laati hain (bar chart).
# State revenue — geographic distribution (Brazil map/state-wise bar chart).

Monthly revenue rows: (23, 2)
Category revenue rows: (71, 2)
State revenue rows: (27, 2)


,order_month,total_revenue
0,2016-09,0.00
1,2016-10,46566.71
2,2016-12,19.62
3,2017-01,127545.67
4,2017-02,271298.65


In [29]:
# Achha result hai — shapes bhi sahi hain (23 months, 71 categories, 27 states — Brazil ke 27 states/union territories se match karta hai).
# Monthly revenue mein 2016-09 aur 2016-12 ka revenue bahut kam hai (0.00 aur 19.62) — yeh Olist dataset ki known characteristic hai,
# jahan business shuru mein bahut chhota tha, phir 2017 se scale hua (jaisa 2017-01 se 2017-02 mein jump dikh raha hai, ₹1.27L se ₹2.71L).
# Yeh data ka genuine pattern hai, error nahi — dashboard mein flag kar sakte ho ki "early months hain sample data, kam volume".

## In teeno tables ko PostgreSQL mein save karo

In [30]:
monthly_revenue.to_sql('monthly_revenue', engine, if_exists='replace', index=False)
category_revenue.to_sql('category_revenue', engine, if_exists='replace', index=False)
state_revenue.to_sql('state_revenue', engine, if_exists='replace', index=False)

print("All 3 summary tables saved to PostgreSQL")

All 3 summary tables saved to PostgreSQL


## Add State in RFM table

In [31]:
# customer_rfm_segments table ko PostgreSQL se wapas load karo
customer_rfm_segments = pd.read_sql("SELECT * FROM customer_rfm_segments;", engine)

# master_df se customer_unique_id -> customer_state ka mapping nikalo (duplicate customers hata ke)
customer_state_map = master_df[['customer_unique_id', 'customer_state']].drop_duplicates(subset='customer_unique_id')

# RFM table mein state merge karo
customer_rfm_segments = customer_rfm_segments.merge(customer_state_map, on='customer_unique_id', how='left')

print("RFM table shape after adding state:", customer_rfm_segments.shape)
customer_rfm_segments.head()

RFM table shape after adding state: (93358, 10)


,customer_unique_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score,segment,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,3,413,Recent High-Value (New Champions),SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3,1,1,311,New Customers,SP
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112,Lost / Churned,SC
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211,At Risk,PA
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214,At Risk,SP


In [32]:
# Expected output: tumhe RFM table dikhegi jisme ab ek naya column customer_state bhi hoga
# (existing columns — segment, recency, frequency waghera — ke saath).

## Category revenue ko state-wise banao

In [33]:
# Category revenue - ab state ke saath break karo
category_revenue_by_state = order_items_df.merge(products_df, on='product_id', how='left') \
    .merge(category_translation_df, on='product_category_name', how='left') \
    .merge(master_df[['order_id', 'customer_state']].drop_duplicates(), on='order_id', how='left') \
    .groupby(['customer_state', 'product_category_name_english'])['price'].sum() \
    .reset_index().sort_values('price', ascending=False)

category_revenue_by_state.columns = ['state', 'category', 'total_revenue']

print("Category-by-state revenue rows:", category_revenue_by_state.shape)
category_revenue_by_state.head()

Category-by-state revenue rows: (1351, 3)


,state,category,total_revenue
1250,SP,bed_bath_table,472238.07
1286,SP,health_beauty,453916.48
1312,SP,watches_gifts,421546.60
1307,SP,sports_leisure,375044.87
1258,SP,computers_accessories,341044.96


## Monthly revenue ko bhi state-wise banao

In [34]:
# Monthly revenue - ab state ke saath break karo
monthly_revenue_by_state = master_df.copy()
monthly_revenue_by_state['order_month'] = monthly_revenue_by_state['order_purchase_timestamp'].dt.to_period('M').astype(str)

monthly_revenue_by_state = monthly_revenue_by_state.groupby(['customer_state', 'order_month'])['payment_value'].sum().reset_index()
monthly_revenue_by_state.columns = ['state', 'order_month', 'total_revenue']

print("Monthly-by-state revenue rows:", monthly_revenue_by_state.shape)
monthly_revenue_by_state.head()

Monthly-by-state revenue rows: (556, 3)


,state,order_month,total_revenue
0,AC,2017-01,723.15
1,AC,2017-02,597.40
2,AC,2017-03,530.18
3,AC,2017-04,1351.51
4,AC,2017-05,2382.64


## teeno naye/updated tables ko PostgreSQL mein save karo

In [35]:
# Teeno tables ko PostgreSQL mein save karo (overwrite karega agar already exist karta hai)
customer_rfm_segments.to_sql('customer_rfm_segments', engine, if_exists='replace', index=False)
category_revenue_by_state.to_sql('category_revenue_by_state', engine, if_exists='replace', index=False)
monthly_revenue_by_state.to_sql('monthly_revenue_by_state', engine, if_exists='replace', index=False)

print("All 3 state-wise tables saved to PostgreSQL successfully!")

All 3 state-wise tables saved to PostgreSQL successfully!


In [36]:
# Ye customer_rfm_segments table ko replace kar dega — matlab purani table (bina state column ke) overwrite ho jayegi naye wale se 
# (state column ke saath). Ye theek hai, koi data loss nahi hoga, bas ek naya column add ho raha hai.